<a href="https://colab.research.google.com/github/himalcodes/gauss-village-travelogue/blob/main/gauss_village_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#The Gauss Village
-Himal Ghimire

In [8]:
# =========================================================
# GAUSS VILLAGE — 3D INTERACTIVE FIGURES (ALL 6)
# =========================================================

import numpy as np
import plotly.graph_objects as go

# =========================================================
# 1. FUNCTION (terrain)
# =========================================================
def f(x, y):
    return np.sin(3*x)*np.cos(3*y)

# grid
n = 60
x = np.linspace(0,1,n)
y = np.linspace(0,1,n)
xx, yy = np.meshgrid(x,y)
Z = f(xx, yy)

# observations
np.random.seed(0)
X = np.random.rand(10,2)
Z_obs = f(X[:,0], X[:,1])

# =========================================================
# HELPER: PLOT 3D SURFACE
# =========================================================
def plot_surface(title, surface_z, obs_points=None, extra=None):

    fig = go.Figure()

    # Main surface
    fig.add_trace(go.Surface(
        z=surface_z,
        x=xx,
        y=yy,
        colorscale='Viridis',
        opacity=0.9,
        name='Surface'
    ))

    # Observations
    if obs_points is not None:
        fig.add_trace(go.Scatter3d(
            x=obs_points[:,0],
            y=obs_points[:,1],
            z=f(obs_points[:,0], obs_points[:,1]),
            mode='markers',
            marker=dict(size=6, color='red'),
            name='Observations'
        ))

    # Optional extras (gradients etc.)
    if extra is not None:
        fig.add_trace(extra)

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Elevation"
        )
    )

    fig.show()

# =========================================================
# FIGURE 1 — GP
# =========================================================
plot_surface(
    "Figure 1: Gaussian Process (Noiseless)",
    Z,
    obs_points=X
)

# =========================================================
# FIGURE 2 — SK (add noise points)
# =========================================================
noise = 0.2
Z_noisy = Z_obs + np.random.normal(0, noise, len(Z_obs))

fig = go.Figure()

fig.add_trace(go.Surface(z=Z, x=xx, y=yy, colorscale='Viridis'))

fig.add_trace(go.Scatter3d(
    x=X[:,0],
    y=X[:,1],
    z=Z_noisy,
    mode='markers',
    marker=dict(size=6, color='orange'),
    name='Noisy Observations'
))

fig.update_layout(title="Figure 2: Stochastic Kriging (Noisy Observations)")
fig.show()

# =========================================================
# FIGURE 3 — GEGP (gradient arrows)
# =========================================================
dx = np.cos(Z_obs)
dy = np.sin(Z_obs)

fig = go.Figure()

fig.add_trace(go.Surface(z=Z, x=xx, y=yy, colorscale='Viridis'))

fig.add_trace(go.Cone(
    x=X[:,0], y=X[:,1], z=Z_obs,
    u=dx, v=dy, w=np.zeros_like(dx),
    sizemode="scaled",
    sizeref=0.2,
    colorscale='Blues',
    name="Gradient"
))

fig.update_layout(title="Figure 3: Gradient-Enhanced GP")
fig.show()

# =========================================================
# FIGURE 4 — GESK (noisy gradients)
# =========================================================
noise_vec = np.random.normal(0,0.5,len(X))

fig = go.Figure()

fig.add_trace(go.Surface(z=Z, x=xx, y=yy, colorscale='Viridis'))

fig.add_trace(go.Cone(
    x=X[:,0], y=X[:,1], z=Z_obs,
    u=dx+noise_vec, v=dy+noise_vec, w=np.zeros_like(dx),
    sizeref=0.2,
    colorscale='Reds',
    name="Noisy Gradient"
))

fig.update_layout(title="Figure 4: GESK (Noisy Gradients)")
fig.show()

# =========================================================
# FIGURE 5 — ACTIVE LEARNING
# =========================================================
X_active = np.random.rand(5,2)

for step in range(3):

    fig = go.Figure()

    fig.add_trace(go.Surface(z=Z, x=xx, y=yy, colorscale='Viridis'))

    fig.add_trace(go.Scatter3d(
        x=X_active[:,0],
        y=X_active[:,1],
        z=f(X_active[:,0], X_active[:,1]),
        mode='markers',
        marker=dict(size=6, color='blue'),
        name='Observed'
    ))

    # next point (random proxy)
    new = np.random.rand(1,2)

    fig.add_trace(go.Scatter3d(
        x=new[:,0], y=new[:,1], z=f(new[:,0], new[:,1]),
        mode='markers',
        marker=dict(size=8, color='yellow'),
        name='Next Point'
    ))

    fig.update_layout(title=f"Figure 5: Active Learning Step {step+1}")
    fig.show()

    X_active = np.vstack([X_active, new])

# =========================================================
# FIGURE 6 — FINAL MAP
# =========================================================
X_final = np.random.rand(25,2)

plot_surface(
    "Figure 6: Final Map",
    Z,
    obs_points=X_final
)